In [1]:
import shutil
from concurrent import futures

import numpy as np
from ase import build
from matplotlib import pyplot as plt
import pyiron_workflow as pwf
from pyiron_workflow_atomistics import engine as engine_mod

from demonstrators import gibbs_nodes, patches

In [ ]:
COVERA = np.sqrt(8.0 / 3.0)
FCC_V_57GPA = 20.7
HCP_V_57GPA = 20.6
HCP_V_84GPA = 19.1
BCC_V_84GPA = 18.9
SPECIES = "Pb"

def fcc_at(volume_per_atom, species=SPECIES):
    return build.bulk(species, "fcc", a=(4.0 * volume_per_atom) ** (1 / 3), cubic=True)


def bcc_at(volume_per_atom, species=SPECIES):
    return build.bulk(species, "bcc", a=(2.0 * volume_per_atom) ** (1 / 3), cubic=True)


def hcp_at(volume_per_atom, covera=COVERA, species=SPECIES):
    # Hexagonal, not orthorhombic: same phonons to 0.03 meV/atom in 200 atoms
    # instead of 240, and it keeps the cell's hexagonal point group so phonopy
    # does not warn. `apply_strains`' c_over_a mode scales the cell rows by
    # (g, g, f), which is exactly the c/a strain on a hexagonal cell.
    a = (4.0 * volume_per_atom / (np.sqrt(3.0) * covera)) ** (1 / 3)
    return build.bulk(species, "hcp", a=a, covera=covera)

## Phase stability

In [ ]:
WORKING_DIR = "scratch_runs"
engine = engine_mod.ASEEngine(
    EngineInput=engine_mod.CalcInputStatic(),
    calculator=patches.PicklableEAM(potential="Pb_II_Wang_2018.eam.alloy", form="alloy"),
    working_directory=WORKING_DIR,
    record_interval=100,
)
temperatures = np.linspace(0.0, 600.0, 7)
pressures_lo = [52.0, 56.0, 58.0, 60.0, 61.0]
pressures_hi = [78.0, 81.0, 84.0, 87.0, 90.0]

gibbs_tolerance = 3e-4
num_points = 5
shape_objective = "static"
max_iter = 12

In [4]:
wf = pwf.Workflow("sweep_lo")
wf.fcc_lo = pwf.node(gibbs_nodes.gibbs_over_pressures)

wf.create_input_for(wf.fcc_lo.inputs.structure, label="fcc_structure_lo")
wf.create_input_for(wf.fcc_lo.inputs.engine)
wf.create_input_for(wf.fcc_lo.inputs.pressures, label="pressures_lo")
wf.create_input_for(wf.fcc_lo.inputs.temperatures)
wf.create_input_for(wf.fcc_lo.inputs.shape_objective)
wf.create_input_for(wf.fcc_lo.inputs.gibbs_tolerance)
wf.create_input_for(wf.fcc_lo.inputs.num_points)
wf.create_input_for(wf.fcc_lo.inputs.max_iterations)
wf.create_input_for(wf.fcc_lo.inputs.working_directory)
wf.create_input_for(wf.fcc_lo.inputs.tag, label="fcc_tag_lo")

wf.hcp_lo = pwf.node(
    gibbs_nodes.gibbs_over_pressures,
    engine=wf.inputs.engine,
    pressures=wf.inputs.pressures_lo,
    temperatures=wf.inputs.temperatures,
    shape_objective=wf.inputs.shape_objective,
    gibbs_tolerance=wf.inputs.gibbs_tolerance,
    num_points=wf.inputs.num_points,
    max_iterations=wf.inputs.max_iterations,
    working_directory=WORKING_DIR,
)
wf.create_input_for(wf.hcp_lo.inputs.structure, label="hcp_structure_lo")
wf.create_input_for(wf.hcp_lo.inputs.tag, label="hcp_tag_lo")

wf.hcp_hi = pwf.node(
    gibbs_nodes.gibbs_over_pressures,
    engine=wf.inputs.engine,
    temperatures=wf.inputs.temperatures,
    shape_objective=wf.inputs.shape_objective,
    gibbs_tolerance=wf.inputs.gibbs_tolerance,
    num_points=wf.inputs.num_points,
    max_iterations=wf.inputs.max_iterations,
    working_directory=WORKING_DIR,
)
wf.create_input_for(wf.hcp_hi.inputs.pressures, label="pressures_hi")
wf.create_input_for(wf.hcp_hi.inputs.structure, label="hcp_structure_hi")
wf.create_input_for(wf.hcp_hi.inputs.tag, label="hcp_tag_hi")

wf.bcc_hi = pwf.node(
    gibbs_nodes.gibbs_over_pressures,
    engine=wf.inputs.engine,
    pressures=wf.inputs.pressures_hi,
    temperatures=wf.inputs.temperatures,
    shape_objective=wf.inputs.shape_objective,
    gibbs_tolerance=wf.inputs.gibbs_tolerance,
    num_points=wf.inputs.num_points,
    max_iterations=wf.inputs.max_iterations,
    working_directory=WORKING_DIR,
)
wf.create_input_for(wf.bcc_hi.inputs.structure, label="bcc_structure_hi")
wf.create_input_for(wf.bcc_hi.inputs.tag, label="bcc_tag_hi")

wf.create_output_from(wf.fcc_lo, label="fcc_sweep_lo")
wf.create_output_from(wf.hcp_lo, label="hcp_sweep_lo")
wf.create_output_from(wf.hcp_hi, label="hcp_sweep_hi")
wf.create_output_from(wf.bcc_hi, label="bcc_sweep_hi")

In [ ]:
wf.validate(do_ontology=True)

In [ ]:
with futures.ProcessPoolExecutor(
        max_workers=2 * (len(pressures_hi) + len(pressures_lo))
) as executor:
    wf.fcc_lo.for_each_0.body.executor = executor
    wf.hcp_lo.for_each_0.body.executor = executor
    wf.hcp_hi.for_each_0.body.executor = executor
    wf.bcc_hi.for_each_0.body.executor = executor
    run = wf.run(
        fcc_structure_lo=fcc_at(HCP_V_57GPA),
        hcp_structure_lo=hcp_at(HCP_V_57GPA),
        hcp_structure_hi=hcp_at(HCP_V_84GPA),
        bcc_structure_hi=bcc_at(HCP_V_84GPA),
        engine=engine,
        pressures_lo=pressures_lo,
        pressures_hi=pressures_hi,
        temperatures=temperatures,
        shape_objective=shape_objective,
        gibbs_tolerance=gibbs_tolerance,
        num_points=num_points,
        max_iterations=max_iter,
        working_directory=WORKING_DIR,
        fcc_tag_lo="fcc_sweep_lo",
        hcp_tag_lo="hcp_sweep_lo",
        hcp_tag_hi="hcp_sweep_hi",
        bcc_tag_hi="bcc_sweep_hi",
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for index, pressure in enumerate(pressures_lo):
    axes[0].plot(
        temperatures,
        (
            run.outputs.hcp_sweep_lo.gibbs[:, index]
            - run.outputs.fcc_sweep_lo.gibbs[:, index]
        ) * 1000.0,
        label=f"{pressure:.0f} GPa",
    )
axes[0].set_title("hcp - fcc")
for index, pressure in enumerate(pressures_hi):
    axes[1].plot(
        temperatures,
        (
            run.outputs.bcc_sweep_hi.gibbs[:, index]
            - run.outputs.hcp_sweep_hi.gibbs[:, index]
        ) * 1000.0,
        label=f"{pressure:.0f} GPa",
    )
axes[1].set_title("bcc - hcp")
for axis in axes:
    axis.axhline(0.0, color="k", lw=0.8)
    axis.set_xlabel("temperature (K)")
    axis.set_ylabel(r"$\Delta G$ (meV/atom)")
    axis.legend(fontsize=8)
plt.show()

In [ ]:
boundary_lo = gibbs_nodes.phase_boundary(
    run.outputs.fcc_sweep_lo,
    run.outputs.hcp_sweep_lo,
)
boundary_hi = gibbs_nodes.phase_boundary(
    run.outputs.hcp_sweep_hi,
    run.outputs.bcc_sweep_hi,
)
print("fcc/hcp boundary (GPa):", np.round(boundary_lo, 2))
print("hcp/bcc boundary (GPa):", np.round(boundary_hi, 2))
# print("hcp/bcc boundary (GPa):", np.round(boundary_hi, 2))

plt.figure(figsize=(6, 5))
plt.plot(boundary_lo, temperatures, "o-", label="fcc / hcp")
plt.plot(boundary_hi, temperatures, "s-", label="hcp / bcc")
plt.xlabel("pressure (GPa)")
plt.ylabel("temperature (K)")
plt.title("Pb phase boundaries (Wang 2018 EAM)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [7]:
shutil.rmtree(WORKING_DIR)